# 03 - Transfer Learning Classification

This notebook trains a MobileNetV2 classifier with ImageNet weights, first with the backbone frozen and then with upper-layer fine-tuning.

## Google Colab Git Setup

Run the next cell only when using Google Colab.


In [1]:
# Colab-only Git setup. Skip this cell when running locally.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/andri-10/computer_vision.git"

REPO_DIR = Path("/content/computer_vision")
PROJECT_DIR = REPO_DIR / "PetVision-DeepLearning"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "origin", "main"])
    else:
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

    os.chdir(PROJECT_DIR)

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"
    ])

    print("Colab repo root:", REPO_DIR)
    print("Colab project root:", os.getcwd())
else:
    print("Not running in Google Colab. Continue with the local setup cells below.")

Colab repo root: /content/computer_vision
Colab project root: /content/computer_vision/PetVision-DeepLearning


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_loader import get_splits, configure_for_performance, save_label_mapping
from src.preprocessing import preprocess_transfer_classification
from src.models_classification import build_transfer_classifier, compile_classifier, unfreeze_for_fine_tuning
from src.training import standard_callbacks
from src.evaluation import collect_predictions, save_classification_outputs, top_k_accuracy, measure_inference_time
from src.visualization import plot_training_history, plot_confusion_matrix

In [4]:
BATCH_SIZE = 32
HEAD_EPOCHS = 3
FINE_TUNE_EPOCHS = 3
BACKBONE = "MobileNetV2"

train_raw, val_raw, test_raw, info, label_names = get_splits()
save_label_mapping(label_names, PROJECT_ROOT / "results/figures/label_mapping.json")

prep = lambda example: preprocess_transfer_classification(example, backbone="mobilenet_v2")
train_ds = configure_for_performance(train_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE, shuffle=True)
val_ds = configure_for_performance(val_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)
test_ds = configure_for_performance(test_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.M6ZDQX_4.0.0/oxford_iiit_pet-train.tfrecord-[0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.M6ZDQX_4.0.0/oxford_iiit_pet-test.tfrecord-[0-9…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [5]:
model, base_model = build_transfer_classifier(input_shape=(224, 224, 3), num_classes=len(label_names), backbone_name=BACKBONE)
compile_classifier(model, learning_rate=1e-3)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "mobilenetv2_pet_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_augmentation     │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 37)             │        47,397 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,305,381 (8.79 MB)

 Trainable params: 47,397 (185.14 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/best_classifier.keras", patience=4),
)
plot_training_history(head_history, PROJECT_ROOT / "results/classification/transfer_head_training_curves.png", "Transfer Head")

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 36s 191ms/step - accuracy: 0.2615 - loss: 2.8073 - top_3_accuracy: 0.4623 - val_accuracy: 0.6671 - val_loss: 1.4107 - val_top_3_accuracy: 0.8927 - learning_rate: 0.0010
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 167ms/step - accuracy: 0.5163 - loss: 1.6579 - top_3_accuracy: 0.7534 - val_accuracy: 0.7595 - val_loss: 0.9721 - val_top_3_accuracy: 0.9293 - learning_rate: 0.0010
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 39s 164ms/step - accuracy: 0.5982 - loss: 1.3581 - top_3_accuracy: 0.8186 - val_accuracy: 0.7880 - val_loss: 0.8528 - val_top_3_accuracy: 0.9389 - learning_rate: 0.0010


In [7]:
unfreeze_for_fine_tuning(base_model)
compile_classifier(model, learning_rate=1e-5)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/best_classifier.keras", patience=4),
)
plot_training_history(fine_tune_history, PROJECT_ROOT / "results/classification/transfer_training_curves.png", "Transfer Fine-tuning")

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 34s 206ms/step - accuracy: 0.6685 - loss: 1.1204 - top_3_accuracy: 0.8665 - val_accuracy: 0.8071 - val_loss: 0.7582 - val_top_3_accuracy: 0.9484 - learning_rate: 1.0000e-05
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 186ms/step - accuracy: 0.6834 - loss: 1.0510 - top_3_accuracy: 0.8770 - val_accuracy: 0.8139 - val_loss: 0.7107 - val_top_3_accuracy: 0.9497 - learning_rate: 1.0000e-05
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 187ms/step - accuracy: 0.7075 - loss: 0.9679 - top_3_accuracy: 0.8893 - val_accuracy: 0.8111 - val_loss: 0.6759 - val_top_3_accuracy: 0.9484 - learning_rate: 1.0000e-05


In [8]:
best_model = tf.keras.models.load_model(PROJECT_ROOT / "models/best_classifier.keras")
test_metrics = best_model.evaluate(test_ds, verbose=1)
print(dict(zip(best_model.metrics_names, test_metrics)))

y_true, y_pred, y_prob = collect_predictions(best_model, test_ds)
transfer_top3 = top_k_accuracy(y_true, y_prob, k=3)
print("Top-3 accuracy:", transfer_top3)
save_classification_outputs(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/transfer")
plot_confusion_matrix(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/transfer_confusion_matrix.png")

115/115 ━━━━━━━━━━━━━━━━━━━━ 14s 101ms/step - accuracy: 0.7907 - loss: 0.7541 - top_3_accuracy: 0.9406
{'loss': 0.7540807127952576, 'compile_metrics': 0.7906786799430847}
Top-3 accuracy: 0.940583265194876


In [16]:
# Model comparison table. Baseline CNN values come from notebook 2.
sample_batch = next(iter(test_ds))[0]
inference_ms = measure_inference_time(best_model, sample_batch, runs=10)
model_size_mb = (PROJECT_ROOT / "models/best_classifier.keras").stat().st_size / (1024 * 1024)
comparison = pd.DataFrame([
    {"model": "Baseline CNN", "input_size": "128x128", "test_accuracy": 0.0362, "top_3_accuracy": 0.0899, "model_size_mb": None, "inference_ms_per_batch": None},
    {"model": BACKBONE, "input_size": "224x224", "test_accuracy": float(np.mean(y_true == y_pred)), "top_3_accuracy": transfer_top3, "model_size_mb": model_size_mb, "inference_ms_per_batch": inference_ms},
])
comparison.to_csv(PROJECT_ROOT / "results/classification/model_comparison_table.csv", index=False)
comparison

,model,input_size,test_accuracy,top_3_accuracy,model_size_mb,inference_ms_per_batch
0,Baseline CNN,128x128,0.036200,0.089900,NaN,NaN
1,MobileNetV2,224x224,0.790679,0.940583,22.429372,134.24186
